In [1]:
import pandas as pd

In [2]:
data = pd.read_csv("autos.csv", delimiter=',' ,encoding='latin-1')

In [3]:
import re

# Sõnad, mille puhul mudeli nimi tavaliselt lõpeb
import re
import unicodedata


# --------------------------------------------------
# 1. Automarkide normaliseerimine
# --------------------------------------------------

MAKE_ALIASES = {
    "volkswagen": ["volkswagen", "vw"],
    "bmw": ["bmw"],
    "mercedes_benz": ["mercedes benz", "mercedes"],
    "audi": ["audi"],
    "opel": ["opel"],
    "ford": ["ford"],
    "renault": ["renault"],
    "peugeot": ["peugeot"],
    "fiat": ["fiat"],
    "seat": ["seat"],
    "skoda": ["skoda"],
    "mazda": ["mazda"],
    "toyota": ["toyota"],
    "smart": ["smart"],
    "citroen": ["citroen"],
    "nissan": ["nissan"],
    "mini": ["mini"],
    "hyundai": ["hyundai"],
    "volvo": ["volvo"],
    "kia": ["kia"],
    "mitsubishi": ["mitsubishi"],
    "honda": ["honda"],
    "alfa_romeo": ["alfa romeo", "alfa"],
    "suzuki": ["suzuki"],
    "porsche": ["porsche"],
    "chevrolet": ["chevrolet"],
    "chrysler": ["chrysler"],
    "dacia": ["dacia"],
    "land_rover": ["land rover"],
    "jeep": ["jeep"],
    "subaru": ["subaru"],
    "daihatsu": ["daihatsu"],
    "jaguar": ["jaguar"],
    "saab": ["saab"],
    "daewoo": ["daewoo"],
    "lancia": ["lancia"],
    "rover": ["rover"],
    "lada": ["lada"],
    "trabant": ["trabant"],
}


# --------------------------------------------------
# 2. Populaarsemate mudelite kataloog
# --------------------------------------------------

POPULAR_MODELS = {
    "volkswagen": """
        golf passat polo tiguan touran sharan caddy transporter
        touareg t-roc t-cross arteon up lupo bora jetta
        scirocco beetle eos fox phaeton corrado
    """.split(),

    "bmw": """
        1er 2er 3er 4er 5er 6er 7er 8er
        x1 x2 x3 x4 x5 x6 x7
        z3 z4 i3 i4 ix
        m3 m4 m5 m6 m8
    """.split(),

    "mercedes_benz": [
        "a klasse", "b klasse", "c klasse", "e klasse",
        "s klasse", "g klasse", "r klasse", "v klasse",
        "cla", "cls", "gla", "glb", "glc", "gle", "gls",
        "ml", "gl", "slk", "clk", "sl",
        "vito", "viano", "sprinter", "190"
    ],

    "audi": """
        a1 a2 a3 a4 a5 a6 a7 a8
        q2 q3 q4 q5 q7 q8
        tt r8 allroad
    """.split(),

    "opel": """
        corsa astra vectra insignia zafira meriva mokka
        adam karl agila antara frontera omega signum
        tigra combo vivaro movano crossland grandland
    """.split(),

    "ford": [
        "fiesta", "focus", "mondeo", "kuga", "c-max",
        "s-max", "galaxy", "transit", "ranger", "puma",
        "ecosport", "fusion", "ka", "b-max", "tourneo",
        "edge"
    ],

    "renault": [
        "clio", "megane", "scenic", "grand scenic", "twingo",
        "captur", "kadjar", "koleos", "laguna", "espace",
        "modus", "kangoo", "master", "trafic", "zoe"
    ],

    "peugeot": """
        107 108 206 207 208 306 307 308 309
        405 406 407 408 508 607
        2008 3008 4008 5008
        partner expert boxer rifter
    """.split(),

    "fiat": [
        "panda", "punto", "grande punto", "500", "500l", "500x",
        "tipo", "bravo", "brava", "stilo", "doblo", "ducato",
        "fiorino", "sedici", "multipla", "croma", "seicento"
    ],

    "seat": """
        ibiza leon altea toledo cordoba arona ateca
        tarraco mii exeo alhambra
    """.split(),

    "skoda": """
        octavia fabia superb yeti kodiaq karoq kamiq
        scala rapid roomster citigo enyaq
    """.split(),

    "mazda": [
        "2", "3", "5", "6", "cx-3", "cx-30", "cx-5",
        "cx-7", "cx-9", "cx-60", "mx-5", "premacy", "demio"
    ],

    "toyota": [
        "yaris", "corolla", "avensis", "auris", "aygo",
        "rav4", "prius", "verso", "c-hr", "land cruiser",
        "hilux", "proace", "celica", "mr2", "camry"
    ],

    "smart": ["fortwo", "forfour", "roadster"],

    "citroen": [
        "c1", "c2", "c3", "c3 picasso", "c3 aircross",
        "c4", "c4 picasso", "c4 cactus", "c5", "c5 aircross",
        "berlingo", "jumpy", "jumper"
    ],

    "nissan": [
        "micra", "qashqai", "juke", "x-trail", "note",
        "navara", "pathfinder", "primera", "almera",
        "leaf", "nv200", "pulsar"
    ],

    "mini": [
        "cooper", "one", "countryman", "clubman",
        "paceman", "coupe", "roadster", "cabrio"
    ],

    "hyundai": [
        "i10", "i20", "i30", "i40", "ix20", "ix35",
        "tucson", "santa fe", "kona", "getz", "coupe",
        "matrix", "elantra", "accent", "bayon"
    ],

    "volvo": """
        v40 v50 v60 v70 v90
        s40 s60 s80 s90
        xc40 xc60 xc70 xc90
        c30 c70 850
    """.split(),

    "kia": """
        ceed rio sportage sorento picanto venga carens
        optima stonic niro soul
    """.split(),

    "mitsubishi": [
        "colt", "lancer", "outlander", "asx", "pajero",
        "space star", "grandis", "carisma", "eclipse cross"
    ],

    "honda": [
        "civic", "accord", "jazz", "cr-v", "hr-v",
        "fr-v", "insight", "cr-z", "prelude"
    ],

    "alfa_romeo": [
        "147", "156", "159", "166", "giulietta",
        "giulia", "stelvio", "mito", "brera", "gt", "spider"
    ],

    "suzuki": [
        "swift", "sx4", "vitara", "grand vitara", "jimny",
        "ignis", "splash", "alto", "baleno", "s-cross"
    ],

    "porsche": [
        "911", "cayenne", "macan", "panamera",
        "boxster", "cayman", "taycan", "924", "944", "928"
    ],

    "chevrolet": [
        "aveo", "spark", "cruze", "orlando", "captiva",
        "lacetti", "matiz", "kalos", "nubira",
        "camaro", "corvette", "trax"
    ],

    "chrysler": [
        "voyager", "grand voyager", "pt cruiser",
        "300c", "300m", "sebring", "crossfire"
    ],

    "dacia": [
        "logan", "sandero", "duster", "lodgy",
        "dokker", "jogger", "spring"
    ],

    "land_rover": [
        "range rover", "range rover sport", "discovery",
        "discovery sport", "defender", "freelander",
        "evoque", "velar"
    ],

    "jeep": [
        "grand cherokee", "cherokee", "compass",
        "renegade", "wrangler", "patriot",
        "commander", "avenger"
    ],

    "subaru": [
        "impreza", "forester", "outback", "legacy",
        "xv", "levorg", "brz", "justy"
    ],

    "daihatsu": [
        "sirion", "terios", "cuore", "charade",
        "materia", "copen", "move"
    ],

    "jaguar": [
        "xf", "xj", "xe", "xk", "s-type", "x-type",
        "f-pace", "e-pace", "i-pace", "f-type"
    ],

    "saab": ["9-3", "9-5", "900", "9000", "9-7x"],

    "daewoo": [
        "matiz", "lanos", "nubira", "kalos",
        "leganza", "tacuma", "espero"
    ],

    "lancia": [
        "ypsilon", "delta", "musa", "lybra",
        "thesis", "phedra", "zeta", "kappa"
    ],

    "rover": [
        "25", "45", "75", "200", "400",
        "600", "800", "streetwise", "cityrover"
    ],

    "lada": [
        "niva", "110", "111", "112",
        "priora", "kalina", "samara", "granta"
    ],

    "trabant": ["601"],
}


# --------------------------------------------------
# 3. Alternatiivsed kirjapildid ja levinud vead
# --------------------------------------------------

MODEL_ALIASES = {
    "volkswagen": {
        "golf3": "golf",
        "golf4": "golf",
        "golf 3": "golf",
        "golf 4": "golf",
        "t5": "transporter",
        "t6": "transporter",
    },

    "skoda": {
        "oktavia": "octavia",
    },

    "ford": {
        "c max": "c-max",
        "s max": "s-max",
    },

    "jeep": {
        "grand cherokee": "grand cherokee",
    },

    "alfa_romeo": {
        "alfa 147": "147",
        "alfa 156": "156",
        "alfa 159": "159",
    },
}


# --------------------------------------------------
# 4. Üldised puhastamise reeglid
# --------------------------------------------------

STOP_WORDS = {
    "overland", "classic", "platinum", "titanium", "sport",
    "comfortline", "limousine", "bastlerfahrzeug", "facelift",
    "klima", "sitzheizg", "sitzheizung", "bluetooth",
    "tdi", "tdci", "tsi", "fsi", "dci", "hdi", "cdi",
    "ecoboost", "edition", "trendline", "highline",
    "ambiente", "ambition", "automatik", "navi"
}

JUNK_WORDS = {
    "verkaufe", "verkauf", "verkaufen", "biete",
    "gepflegter", "gepflegte", "gepflegtes",
    "sehr", "schoener", "schoner", "top",
    "reserviert", "gebrauchtwagen", "new",
    "angebot", "unfallfrei", "guter", "gute",
    "zu"
}


# --------------------------------------------------
# 5. Teksti normaliseerimine
# --------------------------------------------------

def normalize_text(value: str) -> str:
    if value is None:
        return ""

    value = unicodedata.normalize("NFKD", str(value))
    value = value.encode("ascii", "ignore").decode("ascii")

    value = value.casefold()
    value = value.replace("_", " ")

    # Muudame kirjavahemärgid tühikuteks
    value = re.sub(r"[^a-z0-9]+", " ", value)

    return re.sub(r"\s+", " ", value).strip()


# Loome margi sünonüümide kaudu kanonilise margi
MAKE_LOOKUP = {}

for canonical_make, aliases in MAKE_ALIASES.items():
    MAKE_LOOKUP[normalize_text(canonical_make)] = canonical_make

    for alias in aliases:
        MAKE_LOOKUP[normalize_text(alias)] = canonical_make


# Loome mudelite otsimiseks valmis sõnastiku.
# Iga mudeli alias seotakse kanonilise mudelinimega.
MODEL_LOOKUP = {}

for make, models in POPULAR_MODELS.items():
    MODEL_LOOKUP[make] = {}

    for model in models:
        normalized = normalize_text(model)
        MODEL_LOOKUP[make][normalized] = model.lower()

    # Lisame alternatiivsed kirjapildid
    for alias, canonical in MODEL_ALIASES.get(make, {}).items():
        MODEL_LOOKUP[make][normalize_text(alias)] = canonical.lower()


# --------------------------------------------------
# 6. Peamine mudeli eraldamise funktsioon
# --------------------------------------------------

def extract_model(name: str, make: str) -> str:
    """
    Eraldab automudeli kuulutuse nimest.

    name - kuulutuse algne pealkiri
    make - auto mark, näiteks BMW, Volkswagen või Opel

    Tagastab normaliseeritud mudelinime.
    Kui mudelit ei õnnestu tuvastada, tagastab tühja stringi.
    """

    text = normalize_text(name)
    normalized_make = normalize_text(make)

    if not text or not normalized_make:
        return ""

    canonical_make = MAKE_LOOKUP.get(
        normalized_make,
        normalized_make.replace(" ", "_")
    )

    remaining = text

    # Eemaldame margi kuulutuse nimest.
    # Võimaldab leida margi ka juhul, kui pealkirja ees on reklaamtekst.
    brand_aliases = MAKE_ALIASES.get(
        canonical_make,
        [normalized_make]
    )

    brand_positions = []

    for alias in brand_aliases:
        alias = normalize_text(alias)

        pattern = (
            r"(?<![a-z0-9])"
            + re.escape(alias)
            + r"(?![a-z])"
        )

        match = re.search(pattern, text)

        if match:
            brand_positions.append(match)

    if brand_positions:
        # Võtame tekstis kõige varem esineva margi
        brand_match = min(
            brand_positions,
            key=lambda match: match.start()
        )

        remaining = text[brand_match.end():].strip()

    # Piirame otsingu esimese 10 sõnaga,
    # et vältida kuulutuse lõpus olevate juhuslike sõnade leidmist.
    search_text = " ".join(remaining.split()[:10])

    # BMW mudelitähiste käsitlemine:
    # 318i, 320d, 520d, 530d jne.
    if canonical_make == "bmw":
        match = re.search(
            r"(?<![a-z0-9])([1-8]\d{2}[di e]?)(?![a-z0-9])",
            search_text.replace(" ", "")
        )

        if match:
            return match.group(1).replace(" ", "")

    # Mercedes-Benz mudelitähised:
    # C 180, C 200, E 220, A 180 jne.
    if canonical_make == "mercedes_benz":
        match = re.search(
            r"(?<![a-z0-9])([abcesgrv]\s?\d{2,3})(?![a-z0-9])",
            search_text
        )

        if match:
            return match.group(1).strip()

    # Otsime mudelit margipõhisest kataloogist.
    models = MODEL_LOOKUP.get(canonical_make, {})

    matches = []

    for alias, canonical_model in models.items():
        pattern = (
            r"(?<![a-z0-9])"
            + re.escape(alias)
            + r"(?![a-z0-9])"
        )

        match = re.search(pattern, search_text)

        if match:
            matches.append((
                match.start(),
                -len(alias),
                canonical_model
            ))

    if matches:
        # Eelistame:
        # 1. Tekstis kõige varem esinevat mudelit.
        # 2. Sama asukoha korral pikemat mudelinime.
        matches.sort()

        return matches[0][2]

    # --------------------------------------------------
    # 7. Varulahendus tundmatute mudelite jaoks
    # --------------------------------------------------

    # Kui mudelit kataloogis pole, kasutame lihtsamat
    # algse meetodi laadset puhastamist.

    words = remaining.split()

    # Eemaldame algusest tüüpilised kuulutuse sõnad
    while words and words[0] in JUNK_WORDS:
        words.pop(0)

    if not words:
        return ""

    result = []

    for i, word in enumerate(words):

        if word in STOP_WORDS:
            break

        # Mootori suurused, näiteks 1.6 või 2.0,
        # on normaliseerimisel kujul "1 6" ja "2 0".
        if (
            i + 1 < len(words)
            and word.isdigit()
            and words[i + 1].isdigit()
        ):
            break

        # Kere tähised nagu E36 ja F30
        if re.fullmatch(r"[efg]\d{2,3}", word):
            break

        # Eemaldame juhuslikud reklaamtekstide sõnad
        if word in JUNK_WORDS:
            continue

        result.append(word)

        # Tundmatu mudeli puhul ei võta tervet pealkirja kaasa
        if len(result) == 2:
            break

    return " ".join(result)

data['clean_model'] = data.apply(
    lambda row: extract_model(row['name'], row['brand']), axis=1
)

In [4]:
from collections import Counter, defaultdict

# Ilmselgelt mitte mudelinimed.
BAD_START = {
    "andere", "sonstige", "sonstiges",
    "auto", "fahrzeug", "diesel", "benzin",
    "tuv", "tuev", "rot",
    "kombi", "cabrio", "kleinwagen",
    "sportwagen", "sportwagon",
    "sommerfahrzeug", "autoverkauf",
    "gebrauchte", "gebraucht",
    "wegen", "mit", "ich", "meine", "alles",
    "in", "guenstiger", "zuverlaessiges",
    "schmucker", "sparsamer", "vollaustattung",
    "sonderangebot", "selespeed", "jts", "jtd",
    "sportliches",
}

# Mootori-, varustuse- ja muud lisatähised,
# mis ei peaks eraldi mudelikategooriat tekitama.
BAD_WORDS = BAD_START | {
    "jtdm", "tdi", "tsi", "cdi", "hdi", "dci",
    "fsi", "gdi", "ts", "twin", "spark",
    "kw", "ps", "hp", "aut", "automatik",
    "automatic", "leder", "junior",
    "quadrifogli", "selespeed",
    "sport", "line", "comfortline",
    "trendline", "highline", "sline",
    "avant",
    "grau", "rot", "schwarz", "weiss",
    "silber", "blau", "gruen", "braun",
    "gelb", "beige", "metallic", "zustand"
}

COLORS = {
    "grau", "rot", "schwarz", "weiss",
    "silber", "blau", "gruen", "braun",
    "gelb", "beige", "metallic",
}


def normalize_text(value):
    """Muudab teksti ühtsele kujule."""
    if pd.isna(value):
        return ""

    value = str(value).casefold()
    value = unicodedata.normalize("NFKD", value)
    value = "".join(
        char for char in value
        if not unicodedata.combining(char)
    )
    value = value.replace("ß", "ss")
    value = re.sub(r"[^a-z0-9]+", " ", value)

    return " ".join(value.split())


def compact(value):
    """Eemaldab tühikud, et kirjapildi variante võrrelda."""
    return re.sub(r"[^a-z0-9]", "", value)


def format_model(value):
    """Ühtlustab mudelinime kirjapildi."""
    value = normalize_text(value)

    # C 200 -> C200
    value = re.sub(
        r"\b([a-z])\s+(\d+)\b",
        r"\1\2",
        value
    )

    # 328 i -> 328i
    value = re.sub(
        r"\b(\d+)\s+([a-z])\b",
        r"\1\2",
        value
    )

    return value


def strip_brand(model, brand):
    """Eemaldab mudelinime ette sattunud automargi."""
    model_parts = model.split()
    brand_parts = brand.split()

    # Näiteks: alfa romeo 147 -> 147
    if brand_parts and model_parts[:len(brand_parts)] == brand_parts:
        return " ".join(model_parts[len(brand_parts):])

    # Näiteks: alfa 156sw -> 156sw
    if brand_parts and model_parts and model_parts[0] == brand_parts[0]:
        return " ".join(model_parts[1:])

    # Näiteks: alfaromeogtv -> gtv
    model_key = compact(model)
    brand_key = compact(brand)

    if brand_key and model_key.startswith(brand_key):
        return model_key[len(brand_key):]

    return model


def is_model_code(key):
    """Tunneb ära tüüpilise mudelikoodi kuju."""
    return bool(
        re.fullmatch(r"\d{2,3}", key)
        or re.fullmatch(r"\d{1,3}[a-z]", key)
        or re.fullmatch(r"[a-z]\d{1,3}[a-z]?", key)
    )


def is_bad_candidate(display):
    """Kontrollib, kas väärtus sobib mudelinimeks."""
    if not display:
        return True

    tokens = display.split()

    if tokens[0] in BAD_START:
        return True

    if len(tokens) > 2:
        return True

    if any(token in BAD_WORDS for token in tokens):
        return True

    if any(
        re.fullmatch(r"\d+(kw|ps|hp)", token)
        for token in tokens
    ):
        return True

    return False


def clean_car_models(
    data,
    brand_col="brand",
    model_col="clean_model",
    output_col="clean_model_2",
    min_frequency=15
):
    """
    Puhastab automaatselt kõigi automarkide mudelinimed.

    Tagastab:
        cleaned_df: DataFrame koos puhastatud mudelite veeruga.
        unknowns: tabel väärtustest, mida ei suudetud tuvastada.
    """

    prepared = []
    frequencies = Counter()
    forms = defaultdict(Counter)

    # 1. Normaliseerime kõik väärtused.
    for brand, model in zip(data[brand_col], data[model_col]):

        brand_norm = normalize_text(brand)
        model_norm = normalize_text(model)

        model_norm = strip_brand(model_norm, brand_norm)

        brand_key = compact(brand_norm)
        model_key = compact(model_norm)
        display = format_model(model_norm)

        prepared.append((brand_key, model_key, display))

        if model_key:
            frequencies[(brand_key, model_key)] += 1
            forms[(brand_key, model_key)][display] += 1

    # 2. Koostame iga margi mudelikataloogi.
    candidates = defaultdict(dict)

    for (brand, key), count in frequencies.items():

        display = forms[(brand, key)].most_common(1)[0][0]
        tokens = display.split()

        # Säilitame haruldased mudelinimed,
        # millele järgneb ainult värvinimetus.
        if (
            len(tokens) == 2
            and tokens[1] in COLORS
            and re.fullmatch(r"[a-z]{4,}", tokens[0])
            and tokens[0] not in BAD_START
        ):
            candidates[brand].setdefault(
                compact(tokens[0]), tokens[0]
            )

        if is_bad_candidate(display):
            continue

        # Mudel lisatakse, kui see on piisavalt sage
        # või näeb välja nagu tüüpiline mudelikood.
        is_single_word_name = (
            len(tokens) == 1
            and re.fullmatch(r"[a-z]{4,}", tokens[0])
        )

        if (
            count >= min_frequency
            or is_model_code(key)
            or is_single_word_name
        ):
            candidates[brand][key] = display

    # 3. Sobitame kõik ülejäänud väärtused kataloogiga.
    cleaned_models = []

    for brand, key, display in prepared:

        if not key or not display:
            cleaned_models.append("unknown")
            continue

        # Ilmselge kuulutuse tekst.
        if display.split()[0] in BAD_START:
            cleaned_models.append("unknown")
            continue

        brand_candidates = candidates[brand]

        # Täpne vaste.
        if key in brand_candidates:
            cleaned_models.append(brand_candidates[key])
            continue

        # Tüüpiline haruldane mudelikood.
        if is_model_code(key):
            cleaned_models.append(display)
            continue

        # Pikemad mudelinimed enne lühemaid,
        # et näiteks GTV ei läheks GT alla.
        matched = None

        for candidate in sorted(
            brand_candidates,
            key=len,
            reverse=True
        ):
            if len(candidate) < 2:
                continue

            if not key.startswith(candidate):
                continue

            suffix = key[len(candidate):]

            # Väldime näiteks A4 sobitamist A40 külge.
            if (
                suffix
                and candidate[-1].isdigit()
                and suffix[0].isdigit()
            ):
                continue

            matched = brand_candidates[candidate]
            break

        cleaned_models.append(
            matched if matched is not None else "unknown"
        )

    # 4. Säilitame originaalandmed ja lisame uue veeru.
    result = data.copy()
    result[output_col] = cleaned_models

    # 5. Koostame ülevaate väärtustest, mida ei tuvastatud.
    unknowns = (
        result[result[output_col] == "unknown"]
        .groupby([brand_col, model_col], dropna=False)
        .size()
        .rename("count")
        .reset_index()
        .sort_values("count", ascending=False)
    )

    return result, unknowns

data, unknowns = clean_car_models(
    data,
    brand_col="brand",
    model_col="clean_model",
    min_frequency=15
)

print(data["clean_model_2"].value_counts())
print(unknowns.head(10))# Loeme iga margi ja mudeli kombinatsiooni esinemissageduse
model_counts = data.groupby(
    ["brand", "clean_model_2"]
)["clean_model_2"].transform("size")

# Ühendame haruldased mudelid kategooriasse other
rare_models = (
    (model_counts < 3)
    & (data["clean_model_2"] != "unknown")
)

data["model_final"] = data["clean_model_2"]
data.loc[rare_models, "model_final"] = "other"

clean_model_2
golf                   30698
unknown                25143
polo                   13332
corsa                  12604
astra                  10817
                       ...  
twingt                     1
c 300t                     1
angestellte                1
spacelight                 1
leistungssteigerung        1
Name: count, Length: 3117, dtype: int64
              brand clean_model  count
592             bmw               4963
109            audi                689
9696     volkswagen                398
4096  mercedes_benz                364
5637           opel                275
6405        peugeot                259
7776          smart                211
2629           ford                200
6814        renault                189
3859          mazda                141


In [5]:
data.head()

,dateCrawled,name,seller,offerType,price,abtest,vehicleType,yearOfRegistration,gearbox,powerPS,...,fuelType,brand,notRepairedDamage,dateCreated,nrOfPictures,postalCode,lastSeen,clean_model,clean_model_2,model_final
0,2016-03-24 11:52:17,Golf_3_1.6,privat,Angebot,480,test,NaN,1993,manuell,0,...,benzin,volkswagen,NaN,2016-03-24 00:00:00,0,70435,2016-04-07 03:16:57,golf,golf,golf
1,2016-03-24 10:58:45,A5_Sportback_2.7_Tdi,privat,Angebot,18300,test,coupe,2011,manuell,190,...,diesel,audi,ja,2016-03-24 00:00:00,0,66954,2016-04-07 01:46:50,a5,a5,a5
2,2016-03-14 12:52:21,"Jeep_Grand_Cherokee_""Overland""",privat,Angebot,9800,test,suv,2004,automatik,163,...,diesel,jeep,NaN,2016-03-14 00:00:00,0,90480,2016-04-05 12:47:46,grand cherokee,grand cherokee,grand cherokee
3,2016-03-17 16:54:04,GOLF_4_1_4__3TÜRER,privat,Angebot,1500,test,kleinwagen,2001,manuell,75,...,benzin,volkswagen,nein,2016-03-17 00:00:00,0,91074,2016-03-17 17:40:17,golf,golf,golf
4,2016-03-31 17:25:20,Skoda_Fabia_1.4_TDI_PD_Classic,privat,Angebot,3600,test,kleinwagen,2008,manuell,69,...,diesel,skoda,nein,2016-03-31 00:00:00,0,60437,2016-04-06 10:17:21,fabia,fabia,fabia


In [6]:
data = data.drop(columns='postalCode')
data = data.drop(columns='nrOfPictures')

data['dateCreated'] = pd.to_datetime(data['dateCreated'])
data['lastSeen'] = pd.to_datetime(data['lastSeen'])


data['listingAge'] = (data['lastSeen'] - data['dateCreated']) / pd.Timedelta(days=1)

data = data.drop(columns=['dateCreated', 'lastSeen'])

data = data[data.yearOfRegistration <= 2016]
data = data[data.yearOfRegistration >= 1995]

data = data.dropna(subset=['model', 'gearbox', 'fuelType'])

data = data.drop_duplicates()

data = data[data.price > 500]
data = data[data.price < 100000]

data = data[data.offerType == 'Angebot']
data = data.drop(columns = 'offerType')

data = data[data.vehicleType != "andere"]

data = data[data.powerPS > 5]
data = data[data.powerPS < 1000]

data["model_final"] = data["model_final"].replace("fokus", "focus")
data["model_final"] = data["model_final"].replace("pasat", "passat")
data = data[data["model_final"] != "unknown"]


In [7]:
data.head()

,dateCrawled,name,seller,price,abtest,vehicleType,yearOfRegistration,gearbox,powerPS,model,kilometer,monthOfRegistration,fuelType,brand,notRepairedDamage,clean_model,clean_model_2,model_final,listingAge
2,2016-03-14 12:52:21,"Jeep_Grand_Cherokee_""Overland""",privat,9800,test,suv,2004,automatik,163,grand,125000,8,diesel,jeep,NaN,grand cherokee,grand cherokee,grand cherokee,22.533171
3,2016-03-17 16:54:04,GOLF_4_1_4__3TÜRER,privat,1500,test,kleinwagen,2001,manuell,75,golf,150000,6,benzin,volkswagen,nein,golf,golf,golf,0.736308
4,2016-03-31 17:25:20,Skoda_Fabia_1.4_TDI_PD_Classic,privat,3600,test,kleinwagen,2008,manuell,69,fabia,90000,7,diesel,skoda,nein,fabia,fabia,fabia,6.428715
5,2016-04-04 17:36:23,BMW_316i___e36_Limousine___Bastlerfahrzeug__Ex...,privat,650,test,limousine,1995,manuell,102,3er,150000,10,benzin,bmw,ja,316i,316i,316i,2.803553
6,2016-04-01 20:48:51,Peugeot_206_CC_110_Platinum,privat,2200,test,cabrio,2004,manuell,109,2_reihe,150000,8,benzin,peugeot,nein,206,206,206,4.762951


In [8]:
data["car_age"] = 2026 - data.yearOfRegistration

In [9]:
data = data.drop(columns = ["dateCrawled", "name", "seller", "abtest", "model", "monthOfRegistration", "clean_model", "clean_model_2", "yearOfRegistration"])

In [10]:
data.shape

(241038, 11)

In [11]:
df = pd.read_csv("models_cleaned_over_10_instances.csv", delimiter=',' ,encoding='latin-1')

In [12]:
df.head()

,price,vehicleType,gearbox,powerPS,kilometer,fuelType,brand,notRepairedDamage,model_final,car_age
0,9800,suv,automatik,163,125000,diesel,jeep,NaN,grand cherokee,22
1,1500,kleinwagen,manuell,75,150000,benzin,volkswagen,nein,golf,25
2,3600,kleinwagen,manuell,69,90000,diesel,skoda,nein,fabia,18
3,650,limousine,manuell,102,150000,benzin,bmw,ja,316i,31
4,2200,cabrio,manuell,109,150000,benzin,peugeot,nein,206,22


In [13]:
pd.set_option('display.max_rows', None)

In [14]:
data.groupby(data.brand).model_final.value_counts()

brand          model_final         
alfa_romeo     147                       412
               156                       394
               159                       181
               spider                    110
               mito                      110
               gt                         87
               giulietta                  54
               166                        47
               brera                      30
               gtv                        27
               145                         5
               146                         5
               4c                          3
               other                       1
audi           a4                       8277
               a3                       5237
               a6                       5155
               tt                       1135
               a5                        941
               a1                        580
               a8                        473
               q5  

In [15]:
data["model_final"] = data["model_final"].replace("pheaton", "phaeton")
data["model_final"] = data["model_final"].replace("lupe", "lupo")
data["model_final"] = data["model_final"].replace("'colf", "golf")
data["model_final"] = data["model_final"].replace("passst", "passat")
data["model_final"] = data["model_final"].replace("tuareg", "touareg")
data["model_final"] = data["model_final"].replace("golv", "golf")
data["model_final"] = data["model_final"].replace("passau", "passat")
data["model_final"] = data["model_final"].replace("fokus", "focus")
data["model_final"] = data["model_final"].replace("pasat", "passat")
data["model_final"] = data["model_final"].replace("s3 s", "s3")
data["model_final"] = data["model_final"].replace("s5 s", "s5")
#data["model_final"] = data["model_final"].replace("s5 4" "s5")
data["model_final"] = data["model_final"].replace("s5 s", "s5")
data["model_final"] = data["model_final"].replace("s4 b5", "s4")
data["model_final"] = data["model_final"].replace("s4 s", "s4")
data["model_final"] = data["model_final"].replace("citroeen saxo", "saxo")
data["model_final"] = data["model_final"].replace("c8 2", "c8")
data["model_final"] = data["model_final"].replace("escord", "escort")
data["model_final"] = data["model_final"].replace("escord ghia", "escort")
data["model_final"] = data["model_final"].replace("fieste", "fiesta")
data["model_final"] = data["model_final"].replace("fucus", "focus")
data["model_final"] = data["model_final"].replace("modeo", "mondeo")
data["model_final"] = data["model_final"].replace("galxy", "galaxy")
data["model_final"] = data["model_final"].replace("galaxie", "galaxy")
data["model_final"] = data["model_final"].replace("fiasta", "fiesta")
data["model_final"] = data["model_final"].replace("fieasta", "fiesta")
data["model_final"] = data["model_final"].replace("gelaxy", "galaxy")
data["model_final"] = data["model_final"].replace("fista", "fiesta")
data["model_final"] = data["model_final"].replace("jass", "jazz")
data["model_final"] = data["model_final"].replace("mx5 nb", "mx 5")
data["model_final"] = data["model_final"].replace("mx5 na", "mx 5")
data["model_final"] = data["model_final"].replace("mirca", "micra")
data["model_final"] = data["model_final"].replace("mikra", "micra")
data["model_final"] = data["model_final"].replace("micro", "micra")
data["model_final"] = data["model_final"].replace("asta", "astra")
data["model_final"] = data["model_final"].replace("atra", "astra")
data["model_final"] = data["model_final"].replace("coras", "corsa")
data["model_final"] = data["model_final"].replace("corse", "corsa")
data["model_final"] = data["model_final"].replace("ibizia", "ibiza")
data["model_final"] = data["model_final"].replace("ibzia", "ibiza")
data["model_final"] = data["model_final"].replace("t4 multivan", "multivan t4")
data["model_final"] = data["model_final"].replace("t4 caravelle", "caravelle t4")
data["model_final"] = data["model_final"].replace("passt", "passat")
data["model_final"] = data["model_final"].replace("touren", "touran")
data["model_final"] = data["model_final"].replace("turan", "touran")
data["model_final"] = data["model_final"].replace("passart", "passat")
data["model_final"] = data["model_final"].replace("tauran", "touran")
data["model_final"] = data["model_final"].replace("turan", "touran")
data["model_final"] = data["model_final"].replace("bettle", "beatle")

In [17]:
counts = data["model_final"].value_counts()
valid_models = counts[counts >= 10].index
data = data[data["model_final"].isin(valid_models)]
data = data[data["model_final"] != "other"]

In [18]:
data.shape

(238272, 11)

In [19]:
data.to_csv("autos_clean.csv", index=False)